In [ ]:
import numpy as np
import pandas as pd
import os
import csv
import ast
import seaborn as sns
from matplotlib import pyplot as plt

from openfisca_france_indirect_taxation import FranceIndirectTaxationTaxBenefitSystem
from openfisca_france_indirect_taxation.surveys import SurveyScenario
from openfisca_france_indirect_taxation.utils import assets_directory, get_input_data_frame
from openfisca_france_indirect_taxation.examples.utils_example import (
    df_weighted_average_grouped)
from openfisca_france_indirect_taxation.Calage_consommation_bdf import get_inflators_by_year
from openfisca_france_indirect_taxation.projects.TVA.Utils import stacked_bar_plot, double_stacked_bar_plot
from openfisca_france_indirect_taxation.projects.TVA.Reform_TVA_budget_2025 import augmente_tous_les_taux_pass_through_0_8, augmente_tous_les_taux 

In [ ]:
output_path = 'C:/Users/veve1/OneDrive/Documents/IPP/Budget 2025 TVA/Figures_new/Pass_through_08'
output_path_data = 'C:/Users/veve1/OneDrive/Documents/IPP/Budget 2025 TVA/Figures_new/Pass_through_08/Data_pass_through_08'

In [ ]:
simulated_variables = ['depenses_tva_taux_plein',
'depenses_tva_taux_intermediaire',
'depenses_tva_taux_reduit',
'depenses_tva_taux_super_reduit',
'depenses_ht_tva_taux_plein',
'depenses_ht_tva_taux_intermediaire',
'depenses_ht_tva_taux_reduit',
'depenses_ht_tva_taux_super_reduit',
'depenses_tva_exonere',
'depenses_totales',
'depenses_tot',
'tva_total',
'rev_disponible',
 'niveau_de_vie',
 'niveau_vie_decile',
 'decile_indiv_niveau_vie',
 'ocde10',
 'pondmen',
 'pondindiv',
 'agepr',
 'situapr',
 'situacj',
 'situaagr',
 'nactifs',
 'npers',
 'nadultes',
 'nenfants',
 'nenfact',
 'identifiant_menage']

In [ ]:
input_bdf = get_input_data_frame(2017)
input_bdf = input_bdf.loc[input_bdf['rev_disponible'] > 0]

## Préparation des données

###  Calage sur les types de ménage dans Taxipp

In [ ]:
nb_menage_actif_dixieme = pd.read_csv(assets_directory+'/calibration/2025/Nb_actif_10e.csv')
nb_menage_age_dixieme = pd.read_csv(assets_directory+'/calibration/2025/Nb_menage_age_10e.csv')
nb_menage_type_dixieme = pd.read_csv(assets_directory+'/calibration/2025/Nb_menage_type_nbenf_10e.csv')
niveau_vie_dixieme = [9033, 15217, 18578, 21603, 24476, 27480, 30917, 35394, 42436, 74370]

data_calage = {}
for quantile in range(1,11):
    for revenus_menage_actif_positif in ['>0', '==0']:
        if revenus_menage_actif_positif =='>0':
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(nactifs{revenus_menage_actif_positif})'] = float(nb_menage_actif_dixieme.loc[
                    (nb_menage_actif_dixieme['revenus_menage_actif_positif']>0) & (nb_menage_actif_dixieme['quantile']==quantile), 'revenus_menage_actif_positif_nombre_menages'].iloc[0])
        elif revenus_menage_actif_positif =='==0':
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(nactifs{revenus_menage_actif_positif})'] = float(nb_menage_actif_dixieme.loc[
                    (nb_menage_actif_dixieme['revenus_menage_actif_positif']==0) & (nb_menage_actif_dixieme['quantile']==quantile), 'revenus_menage_actif_positif_nombre_menages'].iloc[0])
    for type_age_menage in ['<35', '35-50', '50-64', '>65']:
        if type_age_menage == '<35':
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(agepr{type_age_menage})'] = float(nb_menage_age_dixieme.loc[
                    (nb_menage_age_dixieme['type_age_menage']==0) & (nb_menage_age_dixieme['quantile']==quantile), 'type_age_menage_nombre_menages'].iloc[0])
        if type_age_menage == '35-50':
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(agepr>=35)&(agepr<50)'] = float(nb_menage_age_dixieme.loc[
                    (nb_menage_age_dixieme['type_age_menage']==1) & (nb_menage_age_dixieme['quantile']==quantile), 'type_age_menage_nombre_menages'].iloc[0])
        if type_age_menage == '50-64':
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(agepr>=50)&(agepr<64)'] = float(nb_menage_age_dixieme.loc[
                    (nb_menage_age_dixieme['type_age_menage']==2) & (nb_menage_age_dixieme['quantile']==quantile), 'type_age_menage_nombre_menages'].iloc[0])
        if type_age_menage == '>65':
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(agepr>=65)'] = float(nb_menage_age_dixieme.loc[
                    (nb_menage_age_dixieme['type_age_menage']==3) & (nb_menage_age_dixieme['quantile']==quantile), 'type_age_menage_nombre_menages'].iloc[0])
    for type_menage in [0, 1, 2, 3]:
        if type_menage == 0:
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(nadultes==1)&(nenfants==0)'] = float(nb_menage_type_dixieme.loc[
                    (nb_menage_type_dixieme['type_menage_nbenf']==type_menage) & (nb_menage_type_dixieme['quantile']==quantile), 'type_menage_nbenf_nombre_menages'].iloc[0])
        if type_menage == 1:
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(nadultes>1)&(nenfants==0)'] = float(nb_menage_type_dixieme.loc[
                    (nb_menage_type_dixieme['type_menage_nbenf']==type_menage) & (nb_menage_type_dixieme['quantile']==quantile), 'type_menage_nbenf_nombre_menages'].iloc[0])
        if type_menage == 2:
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(nadultes>1)&(nenfants>0)'] = float(nb_menage_type_dixieme.loc[
                    (nb_menage_type_dixieme['type_menage_nbenf']==type_menage) & (nb_menage_type_dixieme['quantile']==quantile), 'type_menage_nbenf_nombre_menages'].iloc[0])
        if type_age_menage == 3:
            data_calage[f'(decile_indiv_niveau_vie=={quantile})&(nadultes==1)&(nenfants>0)'] = float(nb_menage_type_dixieme.loc[
                    (nb_menage_type_dixieme['type_menage_nbenf']==type_menage) & (nb_menage_type_dixieme['quantile']==quantile), 'type_menage_nbenf_nombre_menages'].iloc[0])
    #data_calage[f'(decile_indiv_niveau_vie=={quantile}) * npers'] = 6500000
    #data_calage[f'(decile_indiv_niveau_vie=={quantile}) * niveau_de_vie'] = niveau_vie_dixieme[quantile-1] * sum(nb_menage_actif_dixieme[nb_menage_actif_dixieme['quantile']==quantile]['revenus_menage_actif_positif_nombre_menages'])

target_entity_count = sum(nb_menage_actif_dixieme['revenus_menage_actif_positif_nombre_menages'])


In [ ]:
calibration_kwargs = {'target_margins_by_variable': data_calage, 'target_entity_count':target_entity_count,
    'parameters': {'method': "hyperbolic sinus", 'alpha': 1.2, 'maxfev': 1000, 'id_variable': 'identifiant_menage', 'entity': 'menage'}}
#target_entity_count permet de modifier le nombre global de ménages, ce qui est utile pour le calage sur le montant de TVA

year = 2025
data_year = 2017
tax_benefit_system = FranceIndirectTaxationTaxBenefitSystem()

survey_scenario = SurveyScenario.create(
    input_data_frame = input_bdf,           # On prend comme entrée la base BdF restreinte aux ménages avec revenus positifs
    calibration_kwargs = calibration_kwargs,
    tax_benefit_system = tax_benefit_system,
    period = year
    )
test = survey_scenario.create_data_frame_by_entity(['pondmen', 'agepr', 'decile_indiv_niveau_vie'])['menage']

###  Calage sur les niveaux de vie dans Taxipp

In [ ]:
from openfisca_france_indirect_taxation.Calage_revenus_bdf import get_coef_calage_niveau_vie

# Décile par individus rev disponible et niveau de vie pour 2025 dans TaxIPP
data = {"decile_indiv_niveau_vie": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'rev_disponible_taxipp' : [16096, 28268, 33790, 38630, 44127, 50111, 56587, 64575, 77033, 136448],
    'niveau_de_vie' : [9033, 15217, 18578, 21603, 24476, 27480, 30917, 35394, 42436, 74370]
}
decile_taxipp = pd.DataFrame(data)
decile_taxipp.set_index('decile_indiv_niveau_vie', inplace = True)

# La base input_bdf reprend les pondérations issues du calage sur marge 
input_bdf = input_bdf.reset_index()
input_bdf.drop(columns = ['index'], axis = 1, inplace = True)
input_bdf['pondmen'] = test['pondmen']

In [ ]:
input_bdf, df_calage = get_coef_calage_niveau_vie(input_bdf, decile_taxipp, 'indiv')
input_bdf = input_bdf.merge(df_calage[['coef_calage','decile_indiv_niveau_vie']], how = 'left', on = 'decile_indiv_niveau_vie')
input_bdf['rev_disponible'] = input_bdf['coef_calage'] * input_bdf['rev_disponible']
input_bdf['niveau_de_vie_bdf'] = input_bdf['rev_disponible'] / input_bdf['ocde10']

### Calage du rendement du point de TVA

Dans Gesta (Trésor-Eco 2025) Analyse de la composition des recettes de TVA, on apprend que: 
- la conso finale des ménages représente 119,2 Md€ sur 212,3 Md€ (soit 56,2 % des recettes de TVA)
- le rendement statique brut d'un point de TVA est estimé à 13,7 Md€
On peut ainsi recaler les dépenses hors-taxes totales sur ces valeurs.

In [ ]:
inflators_by_year = get_inflators_by_year(rebuild = True, year_range = range(2025, 2026), data_year = data_year, input_data_frame = input_bdf)
inflation_kwargs = dict(inflator_by_variable = inflators_by_year[year])

survey_scenario = SurveyScenario.create(
    input_data_frame = input_bdf,        # on prend la base Bdf avec les revenus calés sur TaxIPP et après repondération (calage sur marge) comme entrée pour la simulation
    inflation_kwargs =  inflation_kwargs,
    tax_benefit_system = tax_benefit_system,
    period = year,
    )

In [ ]:
depenses_ht_totales = (survey_scenario.compute_aggregate(variable = 'depenses_ht_tva_taux_plein', use_baseline = True, period = year) +
 survey_scenario.compute_aggregate(variable = 'depenses_ht_tva_taux_intermediaire', use_baseline = True, period = year) +
 survey_scenario.compute_aggregate(variable = 'depenses_ht_tva_taux_reduit', use_baseline = True, period = year) +
 survey_scenario.compute_aggregate(variable = 'depenses_ht_tva_taux_super_reduit', use_baseline = True, period = year)
 )
depenses_ht_totales
tva_total = survey_scenario.compute_aggregate(variable = 'tva_total', use_baseline = True, period = year)
emplois_taxables_tot_hors_APU = 11.4E11 # old: 13E11 (2024), maintenant TVA nette
tva_totale_hors_APU = 212.3 - 30
part_conso_menages = 119.2 / tva_totale_hors_APU # = 0.654 # old: 0.605 (Adrivon et al. (2016) Le modèle d'estimation de la TVA théorique) part hors APU de la conso des ménages dans la TVA totale
coeff = part_conso_menages * emplois_taxables_tot_hors_APU / depenses_ht_totales

In [ ]:
inflators_2025 = { k:v if k in ['loyer_impute','rev_disponible'] else  v*coeff
                      for k,v in inflators_by_year[2025].items()}
inflation_kwargs = dict(inflator_by_variable = inflators_2025)

survey_scenario = SurveyScenario.create(
    input_data_frame = input_bdf,        # on prend la base Bdf avec les revenus calés sur TaxIPP et après repondération (calage sur marge) comme entrée pour la simulation
    inflation_kwargs =  inflation_kwargs,
    baseline_tax_benefit_system = tax_benefit_system,
    reform = augmente_tous_les_taux_pass_through_0_8,
    period = year,
    )

## Effets budgétaires

In [ ]:
recolte_taux_plein = survey_scenario.compute_aggregate(variable='tva_taux_plein', filter_by = 'rev_disponible > 0', difference= True, period = year) * 1e-9
recolte_taux_inter = survey_scenario.compute_aggregate(variable='tva_taux_intermediaire', filter_by = 'rev_disponible > 0', difference= True, period = year) * 1e-9
recolte_taux_reduit = survey_scenario.compute_aggregate(variable='tva_taux_reduit', filter_by = 'rev_disponible > 0', difference= True, period = year) * 1e-9
recolte_taux_super_reduit = survey_scenario.compute_aggregate(variable='tva_taux_super_reduit', filter_by = 'rev_disponible > 0', difference= True, period = year) * 1e-9
total_recolte = recolte_taux_plein + recolte_taux_inter + recolte_taux_reduit + recolte_taux_super_reduit
total_recolte / part_conso_menages

In [ ]:
part_menages = 138.1 / tva_totale_hors_APU 
print(part_menages)
part_entreprises = 41.1 / tva_totale_hors_APU
print(part_entreprises)
part_assos = 3.1 / tva_totale_hors_APU
print(part_assos)
part_APU = 2.3 / 11.4 # Gesta 2025 part des APU dans le rendement de la TVA nette (hors APU)
print(part_APU)

In [ ]:
recettes_menages = total_recolte * part_menages / part_conso_menages
print(recettes_menages)
recettes_entreprises = total_recolte * part_entreprises / part_conso_menages
print(recettes_entreprises)
recettes_assos = total_recolte * part_assos / part_conso_menages
print(recettes_assos)
recettes_APU = total_recolte * part_APU / part_conso_menages
print(recettes_APU)

In [ ]:
recettes_totales = total_recolte/part_conso_menages + recettes_APU 
print(recettes_totales)

## Effets redistributifs 

In [ ]:
survey_scenario.compute_aggregate(variable ='tva_total', use_baseline= True, period = year) * 1e-9

In [ ]:
baseline_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = True, period = year)['menage']
reform_menage   = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = False, period = year)['menage']

In [ ]:
difference_menage = pd.DataFrame()
baseline_variables = ['depenses_tot','rev_disponible','niveau_de_vie','decile_indiv_niveau_vie','ocde10','pondmen',
                      'nactifs','npers','nadultes','nenfants','agepr','identifiant_menage']
difference_menage[baseline_variables] = baseline_menage[baseline_variables]

difference_menage['depenses_totales'] = baseline_menage['depenses_totales'] - reform_menage['depenses_totales']
difference_menage['depenses_tot_par_uc'] = difference_menage['depenses_tot'] / difference_menage['ocde10']
difference_menage['depenses_totales_par_uc'] = difference_menage['depenses_totales'] / difference_menage['ocde10']

liste_taux = ['plein','intermediaire', 'reduit', 'super_reduit']
for taux in liste_taux:
    difference_menage['depenses_tva_taux_{}'.format(taux)] = baseline_menage['depenses_tva_taux_{}'.format(taux)] - reform_menage['depenses_tva_taux_{}'.format(taux)]
    difference_menage['depenses_par_uc_tva_taux_{}'.format(taux)] = difference_menage['depenses_tva_taux_{}'.format(taux)] / difference_menage['ocde10']

difference_menage['pondindiv'] = difference_menage['pondmen'] * difference_menage['npers']

## Les effets d'une hausse de TVA par décile de niveau de vie

In [ ]:
difference_by_decile = df_weighted_average_grouped(dataframe = difference_menage,
                                                   groupe = 'decile_indiv_niveau_vie',
                                                   varlist = ['depenses_tot','depenses_tot_par_uc','depenses_totales_par_uc',
                                                              'rev_disponible','niveau_de_vie'] +
                                                   ['depenses_par_uc_tva_taux_{}'.format(taux) for taux in liste_taux],
                                                   weights = 'pondindiv'
                                                    )
difference_by_decile.loc[:,['pondindiv','npers']] =  difference_menage.groupby('decile_indiv_niveau_vie')[['pondindiv','npers']].sum()

In [ ]:
difference_by_decile['taux_epargne'] = 1 - difference_by_decile['depenses_tot'] / difference_by_decile['rev_disponible']
for taux in liste_taux:
   difference_by_decile['Taux_effort_tva_taux_{}'.format(taux)] = difference_by_decile['depenses_par_uc_tva_taux_{}'.format(taux)] / difference_by_decile['niveau_de_vie'] * 100
difference_by_decile['Taux_effort_total'] = difference_by_decile['depenses_totales_par_uc'] / difference_by_decile['niveau_de_vie'] * 100

In [ ]:
difference_by_decile.to_csv(os.path.join(output_path_data, 'difference_by_decile.csv'), index = True)

In [ ]:
stacked_bar_plot(difference_by_decile,
                 variables = ['Taux_effort_tva_taux_{}'.format(taux) for taux in liste_taux],
                 labels = ['Taux plein','Taux intermédiaire','Taux réduit','Taux super réduit'],
                 title = "Effets de l'augmentation d'un point de TVA",
                 xlabel = 'Déciles de niveau de vie' ,
                 ylabel ='Variation (en % du niveau de vie)',
                 ylim = (-1.4, 0),
                 colors = list(sns.color_palette("Paired")),
                 savefig = False,
                 outfile = 'Taux_effort.pdf',
                #  errors = 'errors'
                 )

#### Décomposition actifs / inactifs

In [ ]:
difference_by_decile_actifs = df_weighted_average_grouped(dataframe = difference_menage.loc[difference_menage['nactifs'] >= 1],
                                                   groupe = 'decile_indiv_niveau_vie',
                                                   varlist = ['depenses_tot','depenses_tot_par_uc','depenses_totales_par_uc',
                                                              'rev_disponible','niveau_de_vie'] +
                                                   ['depenses_par_uc_tva_taux_{}'.format(taux) for taux in liste_taux],
                                                   weights = 'pondindiv'
                                                    )
difference_by_decile_actifs.loc[:,['pondindiv','npers']] =  difference_menage.loc[difference_menage['nactifs'] >= 1].groupby('decile_indiv_niveau_vie')[['pondindiv','npers']].sum()
difference_by_decile_actifs['actifs'] = 1
difference_by_decile_actifs['taux_epargne'] = 1 - difference_by_decile_actifs['depenses_tot'] / difference_by_decile_actifs['rev_disponible']
for taux in liste_taux:
   difference_by_decile_actifs['Taux_effort_tva_taux_{}'.format(taux)] = difference_by_decile_actifs['depenses_par_uc_tva_taux_{}'.format(taux)] / difference_by_decile_actifs['niveau_de_vie'] * 100
difference_by_decile_actifs['Taux_effort_total'] = difference_by_decile_actifs['depenses_totales_par_uc'] / difference_by_decile_actifs['niveau_de_vie'] * 100

In [ ]:
difference_by_decile_actifs[['taux_epargne']]

In [ ]:
stacked_bar_plot(difference_by_decile_actifs,
                 variables = ['Taux_effort_tva_taux_{}'.format(taux) for taux in liste_taux],
                 labels = ['Taux plein','Taux intermédiaire','Taux réduit','Taux super réduit'],
                 title = "Effets de l'augmentation d'un point de TVA - Actifs",
                 xlabel = 'Déciles de niveau de vie' ,
                 ylabel ='Variation (en % du niveau de vie)',
                 ylim = (-1.4, 0),
                 colors = list(sns.color_palette("Paired")),
                 savefig = False,
                 outfile = 'Taux_effort_actifs_only.pdf',
                #  errors = 'errors'
                )

In [ ]:
difference_by_decile_not_actifs = df_weighted_average_grouped(dataframe = difference_menage.loc[difference_menage['nactifs'] < 1],
                                                   groupe = 'decile_indiv_niveau_vie',
                                                   varlist = ['depenses_tot','depenses_tot_par_uc','depenses_totales_par_uc',
                                                              'rev_disponible','niveau_de_vie'] +
                                                   ['depenses_par_uc_tva_taux_{}'.format(taux) for taux in liste_taux],
                                                   weights= 'pondindiv'
                                                    )
difference_by_decile_not_actifs.loc[:,['pondindiv','npers']] =  difference_menage.loc[difference_menage['nactifs'] < 1].groupby('decile_indiv_niveau_vie')[['pondindiv','npers']].sum()
difference_by_decile_not_actifs['actifs'] = 0
difference_by_decile_not_actifs['taux_epargne'] = 1 - difference_by_decile_not_actifs['depenses_tot'] / difference_by_decile_not_actifs['rev_disponible']
for taux in liste_taux:
   difference_by_decile_not_actifs['Taux_effort_tva_taux_{}'.format(taux)] = difference_by_decile_not_actifs['depenses_par_uc_tva_taux_{}'.format(taux)] / difference_by_decile_not_actifs['niveau_de_vie'] * 100
difference_by_decile_not_actifs['Taux_effort_total'] = difference_by_decile_not_actifs['depenses_totales_par_uc'] / difference_by_decile_not_actifs['niveau_de_vie'] * 100

In [ ]:
difference_by_decile_not_actifs[['taux_epargne']]

In [ ]:
stacked_bar_plot(difference_by_decile_not_actifs,
                 variables = ['Taux_effort_tva_taux_{}'.format(taux) for taux in liste_taux],
                 labels = ['Taux plein','Taux intermédiaire','Taux réduit','Taux super réduit'],
                 title = "Effets de l'augmentation d'un point de TVA - Inactifs",
                 xlabel = 'Déciles de niveau de vie' ,
                 ylabel ='Variation (en % du niveau de vie)',
                 ylim = (-1.4, 0),
                 colors = list(sns.color_palette("Paired")),
                 savefig = False,
                 outfile = 'Taux_effort_not_actifs.pdf',
                #  errors = 'errors'
                 )

In [ ]:
difference_by_decile_actifs_not_actifs = pd.concat([difference_by_decile_actifs, difference_by_decile_not_actifs], axis = 0)
difference_by_decile_actifs_not_actifs.to_csv(os.path.join(output_path_data, 'difference_by_decile_actifs_not_actifs.csv'), index = True)

In [ ]:
double_stacked_bar_plot(difference_by_decile_not_actifs,
                        difference_by_decile_actifs,
                        variables = ['Taux_effort_tva_taux_{}'.format(taux) for taux in liste_taux],
                        labels = ['Taux plein','Taux intermédiaire','Taux réduit','Taux super réduit'],
                        title1 = "Effets d'un point de TVA - Inactifs",
                        title2 = "Effets d'un point de TVA - Actifs",
                        xlabel = 'Déciles de niveau de vie' ,
                        ylabel = 'Variation (en % du niveau de vie)',
                        colors = list(sns.color_palette("Paired")),
                        savefig = True,
                        outfile = 'Taux_effort_actifs_not_actifs.pdf')